{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# IMDB Movie Review Sentiment Classification\n",
    "## Using Bernoulli Naive Bayes\n",
    "\n",
    "**Dataset:** Imdb.csv | **Algorithm:** BernoulliNB | **Task:** Binary Text Classification\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "hROEKUWdh0Y2",
   "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "display_name": ".venv",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.14.2"
    },
    "file_extension": ".py",
    "mimetype": "text/x-python",
    "nbconvert_exporter": "python",
    "pygments_lexer": "ipython3",
    "version": "3.14.2"
   }
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 1 — Imports\n",
    "Import all required libraries for data handling, text processing, model training, and evaluation."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "hoh6nfHgh0Y4",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "All libraries imported successfully."
     ]
    }
   ],
   "source": [
    "# Standard library\n",
    "import re\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Data handling\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "\n",
    "# Text preprocessing\n",
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "\n",
    "# Model selection\n",
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "# Naive Bayes classifier\n",
    "from sklearn.naive_bayes import BernoulliNB\n",
    "\n",
    "# Evaluation metric\n",
    "from sklearn.metrics import accuracy_score\n",
    "\n",
    "print(\"All libraries imported successfully.\")"
   ]
  }

In [1]:
# Import all required libraries (in case previous cell wasn't executed)
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

# Load the IMDB dataset using full path
df = pd.read_csv('/Users/reshurajrivansh/genai-training/Python-ML/IMDB Dataset.csv')

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset info:")
df.info()

Dataset shape: (50000, 2)

First 5 rows:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB


# 2. Text Cleaning and Preprocessing

Text data from the web often contains HTML tags, special characters, and inconsistent formatting. We create a clean_text() function that removes HTML tags using BeautifulSoup, converts text to lowercase, removes punctuation and special characters, and normalizes whitespace. This preprocessing step is crucial for effective feature extraction later.

In [2]:
def clean_text(text):
    """
    Clean and preprocess text data by removing HTML tags, 
    converting to lowercase, removing punctuation, and normalizing whitespace.
    """
    # Remove HTML tags using BeautifulSoup
    text = BeautifulSoup(str(text), 'html.parser').get_text()
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and special characters (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace and strip leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply the cleaning function to create a new column
df['cleaned_review'] = df['review'].apply(clean_text)

# Display a sample of original vs cleaned text
print("Sample of original vs cleaned reviews:")
print("Original:", df['review'].iloc[0][:200] + "...")
print("Cleaned:", df['cleaned_review'].iloc[0][:200] + "...")
print(f"\nCleaned reviews column created successfully. No NaN values: {df['cleaned_review'].isnull().sum() == 0}")

Sample of original vs cleaned reviews:
Original: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...
Cleaned: one of the other reviewers has mentioned that after watching just oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brut...

Cleaned reviews column created successfully. No NaN values: True


# 3. Label Encoding

Machine learning models require numerical inputs. We encode the categorical sentiment labels into binary values: 'positive' becomes 1 and 'negative' becomes 0. This encoding allows our Bernoulli Naive Bayes classifier to work with the target variable properly.

In [3]:
# Encode sentiment labels: positive -> 1, negative -> 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Verify the encoding
print("Label distribution:")
print(df['sentiment'].value_counts())
print("\nEncoded label distribution:")
print(df['label'].value_counts())
print(f"\nLabel encoding completed successfully. Unique labels: {df['label'].unique()}")

Label distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Encoded label distribution:
label
1    25000
0    25000
Name: count, dtype: int64

Label encoding completed successfully. Unique labels: [1 0]


# 4. Train-Test Split

We split the dataset into training (80%) and testing (20%) sets to evaluate our model's performance on unseen data. Using random_state=42 ensures reproducibility of the split. This separation prevents data leakage and gives us an unbiased estimate of model performance.

In [4]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_review'], 
    df['label'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['label']  # Ensure balanced split
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Training label distribution: {y_train.value_counts().to_dict()}")
print(f"Test label distribution: {y_test.value_counts().to_dict()}")

Training set size: 40000
Test set size: 10000
Training label distribution: {1: 20000, 0: 20000}
Test label distribution: {0: 5000, 1: 5000}


# 5. Feature Extraction with TF-IDF

We convert text data into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency). This technique captures the importance of words in documents while down-weighting common words. We limit to 5000 features to manage computational complexity and reduce noise. Importantly, we fit only on training data to prevent data leakage.

In [5]:
# Initialize TF-IDF Vectorizer with 5000 maximum features
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit on training data and transform both training and test sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Test TF-IDF matrix shape: {X_test_tfidf.shape}")
print(f"Number of features (vocabulary size): {len(tfidf.vocabulary_)}")
print(f"Sample feature names: {list(tfidf.get_feature_names_out())[:10]}")

Training TF-IDF matrix shape: (40000, 5000)
Test TF-IDF matrix shape: (10000, 5000)
Number of features (vocabulary size): 5000
Sample feature names: ['aaron', 'abandoned', 'abc', 'abilities', 'ability', 'able', 'absence', 'absent', 'absolute', 'absolutely']


# 6. Model Training with Bernoulli Naive Bayes

Bernoulli Naive Bayes is particularly suitable for binary/TF-IDF features as it models the presence/absence of features rather than their counts. We train the classifier on the TF-IDF transformed training data. This algorithm is computationally efficient and works well with text classification tasks.

In [6]:
# Initialize and train the Bernoulli Naive Bayes classifier
model = BernoulliNB()
model.fit(X_train_tfidf, y_train)

print("Bernoulli Naive Bayes model trained successfully!")
print(f"Model classes: {model.classes_}")
print(f"Number of features seen during training: {model.feature_count_.shape[1]}")

Bernoulli Naive Bayes model trained successfully!
Model classes: [0 1]
Number of features seen during training: 5000


# 7. Making Predictions

Using our trained model, we predict sentiment labels for the test set. The model outputs binary predictions (0 for negative, 1 for positive) based on the learned patterns from the training data. These predictions will be compared against the actual labels to evaluate model performance.

In [7]:
# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

print("Predictions completed!")
print(f"Number of predictions: {len(y_pred)}")
print(f"Prediction distribution: {np.bincount(y_pred)}")
print(f"Sample predictions (first 10): {y_pred[:10]}")
print(f"Actual labels (first 10): {y_test.values[:10]}")

Predictions completed!
Number of predictions: 10000
Prediction distribution: [4996 5004]
Sample predictions (first 10): [0 1 1 0 0 0 0 1 0 0]
Actual labels (first 10): [0 0 1 0 0 0 0 0 0 0]


# 8. Model Evaluation

We evaluate the model's performance using accuracy score, which measures the proportion of correct predictions. For this balanced binary classification task, accuracy provides a good overall performance metric. The expected accuracy for BernoulliNB on this dataset is typically around 85-87%.

In [8]:
# Calculate and print the accuracy score
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print(f"Accuracy percentage: {accuracy * 100:.2f}%")
print(f"\nCorrect predictions: {accuracy * len(y_test):.0f} out of {len(y_test)}")
print(f"Incorrect predictions: {(1 - accuracy) * len(y_test):.0f} out of {len(y_test)}")

Model Accuracy: 0.8484
Accuracy percentage: 84.84%

Correct predictions: 8484 out of 10000
Incorrect predictions: 1516 out of 10000
